In [ ]:
import tensorflow as tf

def encoder_block(inputs, num_filters):
    # Używamy padding='same' żeby zachować wymiary
    x = tf.keras.layers.Conv2D(num_filters, 3, padding='same')(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    
    x = tf.keras.layers.Conv2D(num_filters, 3, padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    
    pooled = tf.keras.layers.MaxPool2D(pool_size=(2, 2), strides=2)(x)
    
    return x, pooled  # Zwracamy też skip connection przed poolingiem

In [ ]:
def decoder_block(inputs, skip_features, num_filters):
    # Upsampling
    x = tf.keras.layers.Conv2DTranspose(num_filters, (2, 2), strides=2, padding='same')(inputs)
    
    # Concatenate ze skip connection
    x = tf.keras.layers.Concatenate()([x, skip_features])
    
    # Konwolucje
    x = tf.keras.layers.Conv2D(num_filters, 3, padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    
    x = tf.keras.layers.Conv2D(num_filters, 3, padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)

    return x

In [ ]:
def unet_model(input_shape=(256, 256, 3), num_classes=1):
    inputs = tf.keras.layers.Input(shape=input_shape)
    
    # Contracting Path (Encoder)
    s1, p1 = encoder_block(inputs, 64)   # 256 -> 128
    s2, p2 = encoder_block(p1, 128)      # 128 -> 64
    s3, p3 = encoder_block(p2, 256)      # 64 -> 32
    s4, p4 = encoder_block(p3, 512)      # 32 -> 16
    
    # Bottleneck
    b1 = tf.keras.layers.Conv2D(1024, 3, padding='same')(p4)
    b1 = tf.keras.layers.BatchNormalization()(b1)
    b1 = tf.keras.layers.Activation('relu')(b1)
    b1 = tf.keras.layers.Conv2D(1024, 3, padding='same')(b1)
    b1 = tf.keras.layers.BatchNormalization()(b1)
    b1 = tf.keras.layers.Activation('relu')(b1)
    
    # Expansive Path (Decoder)
    d1 = decoder_block(b1, s4, 512)      # 16 -> 32
    d2 = decoder_block(d1, s3, 256)      # 32 -> 64
    d3 = decoder_block(d2, s2, 128)      # 64 -> 128
    d4 = decoder_block(d3, s1, 64)       # 128 -> 256
    
    # Output - sigmoid dla binarnej segmentacji
    outputs = tf.keras.layers.Conv2D(num_classes, 1, padding='same', activation='sigmoid')(d4)
    
    model = tf.keras.models.Model(inputs=inputs, outputs=outputs, name='U-Net')
    return model

# Test modelu
if __name__ == '__main__':
    model = unet_model(input_shape=(256, 256, 3), num_classes=1)
    model.summary()
    print(f"\nInput shape: {model.input_shape}")
    print(f"Output shape: {model.output_shape}")

In [8]:
# === METODA 1: Przez Google Drive (ZALECANA) ===
# Wrzuć plik data.zip na swój Google Drive, potem uruchom:

from google.colab import drive
import zipfile
import os

# Montuj Google Drive (force_remount wymusi ponowną autoryzację)
drive.mount('/content/drive', force_remount=True)

# Ścieżka do ZIP na Google Drive (zmień jeśli inna lokalizacja)
ZIP_PATH = '/content/drive/MyDrive/INZYNIERKA/DATA/data.zip'

if os.path.exists(ZIP_PATH):
    print(f"Znaleziono: {ZIP_PATH}")
    print("Rozpakowywanie...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall('/content/')
    print("Gotowe! Struktura:")
    !ls -la /content/
else:
    print(f"Nie znaleziono pliku: {ZIP_PATH}")
    print("\nSprawdź czy plik istnieje na Drive:")
    !ls -la "/content/drive/MyDrive/INZYNIERKA/DATA/" 2>/dev/null || echo "Folder nie istnieje. Sprawdź ścieżkę."

KeyboardInterrupt: 

In [ ]:
# === Konfiguracja ścieżek i generatora ===
import numpy as np
from PIL import Image
import glob
import os
import tensorflow as tf

# ZMIEŃ TĘ ŚCIEŻKĘ jeśli struktura ZIP jest inna!
DATA_PATH = '/content/data'  # lub '/content/App/data'

TRAIN_IMG_PATH = os.path.join(DATA_PATH, 'processed/split/train/images/tiled')
TRAIN_MASK_PATH = os.path.join(DATA_PATH, 'processed/split/train/masks/tiled')
VAL_IMG_PATH = os.path.join(DATA_PATH, 'processed/split/val/images/tiled')
VAL_MASK_PATH = os.path.join(DATA_PATH, 'processed/split/val/masks/tiled')

# Parametry
IMG_SIZE = (256, 256)
BATCH_SIZE = 8

# Sprawdź czy ścieżki istnieją
print("Sprawdzanie ścieżek...")
for name, path in [("Train IMG", TRAIN_IMG_PATH), ("Train MASK", TRAIN_MASK_PATH),
                   ("Val IMG", VAL_IMG_PATH), ("Val MASK", VAL_MASK_PATH)]:
    if os.path.exists(path):
        folders = os.listdir(path)
        print(f"✅ {name}: {len(folders)} folderów")
    else:
        print(f"❌ {name}: NIE ISTNIEJE - {path}")

def get_tile_pairs(img_base_dir, mask_base_dir):
    """Zbierz ścieżki do par obraz-maska"""
    pairs = []
    
    if not os.path.exists(img_base_dir):
        print(f"❌ Folder nie istnieje: {img_base_dir}")
        return pairs
    
    img_folders = [f for f in os.listdir(img_base_dir) if os.path.isdir(os.path.join(img_base_dir, f))]
    
    for img_folder in img_folders:
        mask_folder = img_folder + "_mask"
        
        img_folder_path = os.path.join(img_base_dir, img_folder)
        mask_folder_path = os.path.join(mask_base_dir, mask_folder)
        
        if not os.path.exists(mask_folder_path):
            continue
        
        tile_files = glob.glob(os.path.join(img_folder_path, '*.png'))
        
        for tile_path in tile_files:
            tile_name = os.path.basename(tile_path)
            mask_path = os.path.join(mask_folder_path, tile_name)
            
            if os.path.exists(mask_path):
                pairs.append((tile_path, mask_path))
    
    return pairs

class TileDataGenerator(tf.keras.utils.Sequence):
    """Generator wczytujący dane partiami - oszczędza RAM"""
    
    def __init__(self, pairs, batch_size=8, shuffle=True, **kwargs):
        super().__init__(**kwargs)  # Ważne dla Keras 3!
        self.pairs = pairs
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indexes = np.arange(len(self.pairs))
        if self.shuffle:
            np.random.shuffle(self.indexes)
    
    def __len__(self):
        return max(1, int(np.ceil(len(self.pairs) / self.batch_size)))
    
    def __getitem__(self, index):
        if len(self.pairs) == 0:
            # Zwróć puste tensory jeśli brak danych
            return np.zeros((1, 256, 256, 3)), np.zeros((1, 256, 256, 1))
        
        batch_indexes = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]
        batch_pairs = [self.pairs[i] for i in batch_indexes]
        
        images = []
        masks = []
        
        for img_path, mask_path in batch_pairs:
            img = Image.open(img_path).convert('RGB')
            img = np.array(img) / 255.0
            images.append(img)
            
            mask = Image.open(mask_path).convert('L')
            mask = np.array(mask) / 255.0
            mask = mask[..., np.newaxis]
            masks.append(mask)
        
        return np.array(images, dtype=np.float32), np.array(masks, dtype=np.float32)
    
    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

# Zbierz pary ścieżek
print("\nZbieranie ścieżek do tile'ów...")
train_pairs = get_tile_pairs(TRAIN_IMG_PATH, TRAIN_MASK_PATH)
val_pairs = get_tile_pairs(VAL_IMG_PATH, VAL_MASK_PATH)

print(f"✅ Dane treningowe: {len(train_pairs)} par tile'ów")
print(f"✅ Dane walidacyjne: {len(val_pairs)} par tile'ów")

if len(train_pairs) == 0:
    print("\n⚠️ UWAGA: Brak danych treningowych!")
    print("Sprawdź czy:")
    print("1. ZIP został poprawnie rozpakowany")
    print("2. Ścieżka DATA_PATH jest poprawna")
    print("3. Foldery tiled/ istnieją i zawierają pliki .png")
else:
    # Utwórz generatory
    train_generator = TileDataGenerator(train_pairs, batch_size=BATCH_SIZE, shuffle=True)
    val_generator = TileDataGenerator(val_pairs, batch_size=BATCH_SIZE, shuffle=False)
    
    print(f"\nBatchy treningowych: {len(train_generator)}")
    print(f"Batchy walidacyjnych: {len(val_generator)}")
    
    # Test - wczytaj jeden batch
    print("\nTest wczytywania batcha...")
    X_test, y_test = train_generator[0]
    print(f"Batch shape: X={X_test.shape}, y={y_test.shape}")

In [ ]:
# === Kompilacja i trening modelu ===
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

# Utwórz model
model = unet_model(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3), num_classes=1)

# Kompilacja modelu
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.IoU(num_classes=2, target_class_ids=[1], name='iou')]
)

# Callbacki
callbacks = [
    # Zapisz najlepszy model
    ModelCheckpoint(
        '/content/drive/MyDrive/INZYNIERKA/models/best_unet.h5',
        monitor='val_iou',
        mode='max',
        save_best_only=True,
        verbose=1
    ),
    # Zatrzymaj jeśli brak poprawy
    EarlyStopping(
        monitor='val_iou',
        mode='max',
        patience=10,
        verbose=1
    ),
    # Zmniejsz learning rate gdy plateau
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    )
]

# Utwórz folder na modele
import os
os.makedirs('/content/drive/MyDrive/INZYNIERKA/models', exist_ok=True)

print("Model skompilowany. Gotowy do treningu!")

In [ ]:
# === TRENING z generatorem ===
EPOCHS = 50

history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

print("\n✅ Trening zakończony!")

In [ ]:
# === Wizualizacja wyników treningu ===
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss
axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

# Accuracy
axes[1].plot(history.history['accuracy'], label='Train Acc')
axes[1].plot(history.history['val_accuracy'], label='Val Acc')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

# IoU
axes[2].plot(history.history['iou'], label='Train IoU')
axes[2].plot(history.history['val_iou'], label='Val IoU')
axes[2].set_title('IoU (Intersection over Union)')
axes[2].set_xlabel('Epoch')
axes[2].legend()

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/INZYNIERKA/models/training_history.png', dpi=150)
plt.show()

print(f"\nNajlepsza walidacyjna IoU: {max(history.history['val_iou']):.4f}")

In [ ]:
# === Testowanie na zbiorze testowym ===
TEST_IMG_PATH = os.path.join(DATA_PATH, 'processed/split/test/images/tiled')
TEST_MASK_PATH = os.path.join(DATA_PATH, 'processed/split/test/masks/tiled')

print("Zbieranie ścieżek testowych...")
test_pairs = get_tile_pairs(TEST_IMG_PATH, TEST_MASK_PATH)
print(f"✅ Dane testowe: {len(test_pairs)} par tile'ów")

test_generator = TileDataGenerator(test_pairs, batch_size=BATCH_SIZE, shuffle=False)

# Wczytaj najlepszy model
best_model = tf.keras.models.load_model(
    '/content/drive/MyDrive/INZYNIERKA/models/best_unet.h5',
    custom_objects={'iou': tf.keras.metrics.IoU(num_classes=2, target_class_ids=[1])}
)

# Ewaluacja
results = best_model.evaluate(test_generator, verbose=1)
print(f"\n📊 Wyniki na zbiorze testowym:")
print(f"   Loss: {results[0]:.4f}")
print(f"   Accuracy: {results[1]:.4f}")
print(f"   IoU: {results[2]:.4f}")

In [ ]:
# === Wizualizacja predykcji ===
import matplotlib.pyplot as plt

# Weź jeden batch do wizualizacji
X_sample, y_sample = test_generator[0]

# Predykcje
predictions = best_model.predict(X_sample)

# Wyświetl przykłady (max 5)
n_examples = min(5, len(X_sample))
fig, axes = plt.subplots(n_examples, 3, figsize=(12, 4*n_examples))

for i in range(n_examples):
    # Oryginalny obraz
    axes[i, 0].imshow(X_sample[i])
    axes[i, 0].set_title('Obraz wejściowy')
    axes[i, 0].axis('off')
    
    # Ground truth maska
    axes[i, 1].imshow(y_sample[i].squeeze(), cmap='gray')
    axes[i, 1].set_title('Maska rzeczywista')
    axes[i, 1].axis('off')
    
    # Predykcja
    pred = (predictions[i].squeeze() > 0.5).astype(np.uint8)
    axes[i, 2].imshow(pred, cmap='gray')
    axes[i, 2].set_title('Predykcja modelu')
    axes[i, 2].axis('off')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/INZYNIERKA/models/predictions_examples.png', dpi=150)
plt.show()